In [479]:
import numpy as np
from LiouvilleLanczos.Quantum_computer.Hamiltonian import Line_Hubbard, BoundaryCondition
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.quantum_info import SparsePauliOp
from pauliarray.pauli.weighted_pauli_array import WeightedPauliArray
from pauliarray.pauli.pauli_array import PauliArray
from pauliarray.binary import symplectic
import pauliarray.pauli.pauli_array as p
from pauliarray.binary import bit_operations as bitops
mapper = JordanWignerMapper()


### Conversion methods from fermionicOP to SparsePauliOP to PauliArray

In [480]:
def fermionic_op_to_matrix(op):
    qubit_op = mapper.map(op)
    return np.asarray(qubit_op.to_matrix(), dtype=complex)

def to_sparse_pauli(op):
    """
    Convert FermionicOp -> SparsePauliOp.
    Leave SparsePauliOp unchanged.
    """
    if isinstance(op, SparsePauliOp):
        return op

    if isinstance(op, FermionicOp):
        return mapper.map(op).simplify(atol=1e-12)

    raise TypeError(f"Unsupported operator type: {type(op)}")

def to_pauli(op):
    labels = op.paulis.to_labels()
    weights = op.coeffs
    return WeightedPauliArray.from_labels_and_weights(labels=labels, weights=weights), PauliArray.from_labels(labels=labels)

In [481]:
U = 4 
Ham = Line_Hubbard(-1,U/2,U,3,boundary_condition=BoundaryCondition.OPEN)

H = to_sparse_pauli(Ham)

$$S_x=\dfrac{1}{2}\sum_{i=0}^2\left(c_{i\uparrow}^\dag c_{i\downarrow}+c_{i\downarrow}^\dag c_{i\uparrow}\right) \qquad S_y=\dfrac{1}{2i}\sum_{i=0}^2\left(c_{i\uparrow}^\dag c_{i\downarrow}-c_{i\downarrow}^\dag c_{i\uparrow}\right) \qquad S_z = \dfrac{1}{2}\left(n_{i\uparrow}-n_{i\downarrow}\right)$$

In [482]:
num_spin_orbitals = 6

Sz = FermionicOp(
    {
        "+_0 -_0":  0.5,
        "+_1 -_1": -0.5,

        "+_2 -_2":  0.5,
        "+_3 -_3": -0.5,

        "+_4 -_4":  0.5,
        "+_5 -_5": -0.5,
    },
    num_spin_orbitals=num_spin_orbitals,
)

Sx = FermionicOp(
    {
        "+_0 -_1": 0.5,
        "+_1 -_0": 0.5,

        "+_2 -_3": 0.5,
        "+_3 -_2": 0.5,

        "+_4 -_5": 0.5,
        "+_5 -_4": 0.5,
    },
    num_spin_orbitals=num_spin_orbitals,
)

Sy = FermionicOp(
    {
        "+_0 -_1": -0.5j,
        "+_1 -_0":  0.5j,

        "+_2 -_3": -0.5j,
        "+_3 -_2":  0.5j,

        "+_4 -_5": -0.5j,
        "+_5 -_4":  0.5j,
    },
    num_spin_orbitals=num_spin_orbitals,
)

In [483]:
generators = [Sx,Sy,Sz]
Pgenerators = [0,0,0]
gen_labels = [0,0,0]
for i in range(len(generators)):
    generators[i]= to_sparse_pauli(generators[i])
    Pgenerators[i], gen_labels[i]= to_pauli(generators[i])

SPH = to_sparse_pauli(Ham)
PH, PH_label = to_pauli(SPH)


## Below is the SparsePauliOP commutation demonstration

In [484]:
def commutator(A, B):
    comm = (A @ B - B @ A).simplify(atol=1e-12)
    return np.linalg.norm(comm.to_matrix())
def anticommutator(A,B):
    comm = (A @ B + B @ A).simplify(atol=1e-12)
    return np.linalg.norm(comm.to_matrix())

In [485]:
for gen in generators:
    print(commutator(gen,SPH))


0.0
0.0
0.0


## Generators dont commute or anticommute

In [486]:
for i in range(len(generators)):
    for j in range(i+1,len(generators)):
        print(commutator(generators[i],generators[j]))
        print(anticommutator(generators[i],generators[j]))

4.898979485566356
4.898979485566356
4.898979485566356
4.898979485566356
4.898979485566356
4.898979485566356


## Below is the pauliarray commutation demonstration.

In [487]:
from collections import defaultdict

def pauli_commutator(P1,P2, duplicate = True):
    result = {}
    for A_name, A in P1.items():
        for B_name, B in P2.items():
            key = tuple(sorted([A_name, B_name]))
            if not duplicate and A_name == B_name:
                continue
            if key in result:
                continue
            A_paulis = A.paulis
            B_paulis = B.paulis
            if hasattr(A,"weights") and hasattr (B, "weights"):
                A_coeffs = np.asarray(A.weights, dtype=complex).reshape(-1)
                B_coeffs = np.asarray(B.weights, dtype=complex).reshape(-1)
            else:
                A_coeffs = np.ones(len(A_paulis.to_labels()))
                B_coeffs = np.ones(len(B_paulis.to_labels()))

            dic = defaultdict(complex)

            for i in range(len(A_paulis.to_labels())):
                a_val = A_paulis[i]
                Ca = A_coeffs[i]
                for j in range(len(B_paulis.to_labels())):
                    b_val = B_paulis[j]
                    Cb = B_coeffs[j]
                    comm, coeffs = p.commutator(a_val,b_val)
                    dic[comm.to_labels()[0]] += coeffs*Ca*Cb
            dic = {
                label: coeff
                for label, coeff in dic.items()
                if abs(coeff) > 1e-12
            }
            result[tuple(sorted([A_name, B_name]))]=(len(dic) == 0)
    return result

def pauli_anticommutator(P1,P2, duplicate = True):
    result = {}
    for A_name, A in P1.items():
        for B_name, B in P2.items():
            key = tuple(sorted([A_name, B_name]))
            if not duplicate and A_name == B_name:
                continue
            if key in result:
                continue
            A_paulis = A.paulis
            B_paulis = B.paulis
            if hasattr(A,"weights") and hasattr (B, "weights"):
                A_coeffs = np.asarray(A.weights, dtype=complex).reshape(-1)
                B_coeffs = np.asarray(B.weights, dtype=complex).reshape(-1)
            else:
                A_coeffs = np.ones(len(A_paulis.to_labels()))
                B_coeffs = np.ones(len(B_paulis.to_labels()))

            dic = defaultdict(complex)

            for i in range(len(A_paulis.to_labels())):
                a_val = A_paulis[i]
                Ca = A_coeffs[i]
                for j in range(len(B_paulis.to_labels())):
                    b_val = B_paulis[j]
                    Cb = B_coeffs[j]
                    comm, coeffs = p.anticommutator(a_val,b_val)
                    dic[comm.to_labels()[0]] += coeffs*Ca*Cb
            dic = {
                label: coeff
                for label, coeff in dic.items()
                if abs(coeff) > 1e-12
            }
            result[tuple(sorted([A_name, B_name]))]=(len(dic) == 0)
    return result


In [488]:
ops = {"Sx":Pgenerators[0], "Sy":Pgenerators[1], "Sz":Pgenerators[2]}
hamiltonian = {"H":PH}

for gen in Pgenerators:
    print(gen.paulis.to_labels())

result = pauli_commutator(hamiltonian,ops)

print(result)


['IIIIYY' 'IIIIXX' 'IIYYII' 'IIXXII' 'YYIIII' 'XXIIII']
['IIIIXY' 'IIIIYX' 'IIXYII' 'IIYXII' 'XYIIII' 'YXIIII']
['IIIIIZ' 'IIIIZI' 'IIIZII' 'IIZIII' 'IZIIII' 'ZIIIII']
{('H', 'Sx'): True, ('H', 'Sy'): True, ('H', 'Sz'): True}


## generators dont commute or anticommute

In [489]:
commute = pauli_commutator(ops,ops, duplicate=False)
anti = pauli_anticommutator(ops,ops, duplicate=False)


print("commute: ",commute)
print("anticommute: ", anti)

commute:  {('Sx', 'Sy'): False, ('Sx', 'Sz'): False, ('Sy', 'Sz'): False}
anticommute:  {('Sx', 'Sy'): False, ('Sx', 'Sz'): False, ('Sy', 'Sz'): False}


## Symplectic gram schmidt to cast the symmetry onto one qubit (small demo)

### helpers to use pauliarray symplectic module

In [490]:
def zx_to_pauliarray(zx):
    """
    Convert a binary symplectic zx matrix back into a PauliArray.

    Parameters
    ----------
    zx : array-like, shape (k, 2*n) or (2*n,)
        Binary symplectic representation of Pauli strings.
        First n columns are Z bits, last n columns are X bits.

    Returns
    -------
    paulis : PauliArray
        PauliArray object containing the corresponding Pauli strings.
    """
    zx = np.asarray(zx, dtype=bool)

    # If one Pauli string was passed as shape (2*n,), make it shape (1, 2*n)
    if zx.ndim == 1:
        zx = zx[None, :]

    if zx.shape[1] % 2 != 0:
        raise ValueError("zx must have an even number of columns: 2*n.")

    n = zx.shape[1] // 2

    z_strings = zx[:, :n]
    x_strings = zx[:, n:]

    return PauliArray(z_strings, x_strings)

def canonical_Z_generators(k, n):
    z = np.zeros((k, n), dtype=bool)
    x = np.zeros((k, n), dtype=bool)

    for i in range(k):
        z[i, i] = True

    return symplectic.merge_zx_strings(z, x)


def canonical_X_generators(k, n):
    z = np.zeros((k, n), dtype=bool)
    x = np.zeros((k, n), dtype=bool)

    for i in range(k):
        x[i, i] = True

    return symplectic.merge_zx_strings(z, x)


def inv(A):
    A = np.asarray(A, dtype=bool)

    n, m = A.shape
    
    aug = np.concatenate([A.copy(), np.eye(n, dtype=bool)], axis=1)

    row = 0
    for col in range(n):
        pivots = np.where(aug[row:, col])[0]

        if len(pivots) == 0:
            raise ValueError("Matrix is not invertible over GF(2).")

        pivot = row + pivots[0]

        if pivot != row:
            aug[[row, pivot]] = aug[[pivot, row]]

        for r in range(n):
            if r != row and aug[r, col]:
                aug[r] ^= aug[row]

        row += 1

    return aug[:, n:]

def standard_symplectic_basis(n):
    """
    Return standard binary symplectic basis in zx convention.

    Rows are:
        Z0, Z1, ..., Z_{n-1},
        X0, X1, ..., X_{n-1}

    Shape is (2*n, 2*n).
    """
    Z = np.zeros((n, 2*n), dtype=bool)
    X = np.zeros((n, 2*n), dtype=bool)

    # First n columns are Z bits
    Z[np.arange(n), np.arange(n)] = True

    # Last n columns are X bits
    X[np.arange(n), n + np.arange(n)] = True

    return np.vstack([Z, X])



### Wikipedia:
define $W$ as a linear subspace of a symplectic vector space $V$:

such that $W_\perp = \{v\in V | \omega(v,w)=0 \text{ }\forall \text{ } w\in W\}$

- $W$ is symplectic if $W_\perp \cap W = \{0\}$
- $W$ is isotropic if $W ⊆ W\perp$ 
- $W$ is coisotropic if $W_\perp ⊆ W$ 
- $W$ is Lagrangian if $W = W_\perp$

In [491]:
# print(symplectic.is_orthogonal(zx0,zx1))            ##DO THEY COMMUTE
# print(symplectic.is_coisotropic(np.array([zx0,zx1])))  ## Does it contain its own orthogonal complement?
# print(symplectic.is_isotropic(np.array([zx0,zx1])))     ##Every vector is orthogonal (commutes) to every other vector
# print(symplectic.orthogonal_complement(np.array([zx0,zx1])))    ##returns a set of all binary vectors that are orthogonal to the input vectors
# print(symplectic.isotropic_subspace(np.array([zx0,zx1])))       ## returns an independent basis for the (already orthogonal) subspace
# print(symplectic.coisotropic_subspace(np.array([zx0,zx1])))     ## returns the orthogonal complement of the isotropic subspace
# print(symplectic.lagrangian_subspace(np.array([zx0,zx1])))      ## returns the lagrangian subspace: a maximal set of independent mutually commuting paulis.
# print(symplectic.lagrangian_bitwise_colagrangian_subspaces(np.array([zx0,zx1]))) ## colagrangian subspace (orthogonal subspace)


# print(symplectic.is_lagrangian(np.array([zx0,zx1])))    ## is both isotropic and coisotropic

# print(symplectic.is_diagonal_assigned())        ##

In [492]:
GG = []

for gen in Pgenerators:
    print(gen.paulis.to_labels())
    zx = gen.paulis.zx_strings
    GG.append(zx)

GG = np.vstack(GG).astype(bool)     #stack zx strings to make one big zx-matrix instead of individual zx strings

['IIIIYY' 'IIIIXX' 'IIYYII' 'IIXXII' 'YYIIII' 'XXIIII']
['IIIIXY' 'IIIIYX' 'IIXYII' 'IIYXII' 'XYIIII' 'YXIIII']
['IIIIIZ' 'IIIIZI' 'IIIZII' 'IIZIII' 'IZIIII' 'ZIIIII']


We find a (symplectic) orthogonal basis first using the gram_schmidt_orthogonalization (Full symplectic space in a clean basis)

In [493]:
G_ortho = symplectic.gram_schmidt_orthogonalization(np.array(GG))

Then we find an isotropic subspace which is the subspace in which every pair of vectors has symplectic product equal to 0 (commutes)

In [494]:
G_iso = symplectic.isotropic_subspace(G_ortho)

Next, we find the conjugate subspace, which gives us the anticommuting h to each g in our isotropic subspace. The Union of these two spaces is our original symplectic space

In [495]:
H_conj = symplectic.conjugate_subspace(G_iso)

Now after stacking them on top of each other $B^\text{T} = \{g_1,\cdots,g_k,h_1,\cdots,h_k\}$ forms a cannonical symplectic basis

In [496]:
B = np.vstack([G_iso, H_conj])

The pairing matrix below is analagous to the diagonalized matrix in a euclidean geometry

In [497]:
def pairing_matrix(A, C=None):
    if C is None:
        C = A

    A = np.asarray(A, dtype=bool)
    C = np.asarray(C, dtype=bool)

    M = np.zeros((A.shape[0], C.shape[0]), dtype=int)

    for i in range(A.shape[0]):
        for j in range(C.shape[0]):
            M[i, j] = int(symplectic.dot(A[i], C[j]) % 2)

    return M

In [498]:
pairing = pairing_matrix(B,B)

print(pairing)

[[0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0]]


In [499]:
G_ortho_pauli = zx_to_pauliarray(G_ortho)
B_pauli = zx_to_pauliarray(B)



print(G_ortho_pauli.to_labels())
print(B_pauli.to_labels())

['YYIIII' 'IIYYII' 'IIIIYY' 'IIIIZZ' 'IIZZII' 'ZZIIII' 'IIIIZZ' 'IIZZII'
 'ZZIIII' 'IIIIZZ' 'IIIIII' 'IIZZII' 'IIIIII' 'ZZIIII' 'IIIIII']
['IIIIZZ' 'IIIIXX' 'IIZZII' 'IIXXII' 'ZZIIII' 'XXIIII' 'IIIIIX' 'IIIIZI'
 'IIIXII' 'IIZIII' 'IXIIII' 'ZIIIII']


Now we define our target basis such that $BC = B_\text{target}$ so that we can recover the clifford s.t. $C = B^{-1}B_\text{target}$

In [500]:
n = G_iso.shape[1] // 2
B_target = standard_symplectic_basis(n)

bitwise multiply $B^{-1}B_\text{target}$ to find our clifford

In [501]:
C = bitops.matmul(inv(B),B_target)

print(np.array_equal(bitops.matmul(B, C), B_target))

True


Multiply our original stacked orthogonal symplectic basis B with the clifford we just found, and we find that we are in the standard symplectic basis that we defined above

In [ ]:
CBC = zx_to_pauliarray(bitops.matmul(B,C))

zx_PH = PH.zx_strings
CHC = zx_to_pauliarray()

['IIIIIZ' 'IIIIZI' 'IIIZII' 'IIZIII' 'IZIIII' 'ZIIIII' 'IIIIIX' 'IIIIXI'
 'IIIXII' 'IIXIII' 'IXIIII' 'XIIIII']


In [ ]:
print(CSC.to_labels())